# Initial descriptor calculations, all std RDKit ones, plus the Maccs fingerprints
2025-10-28, Alexander Minidis

Update: 11/1 w maccs and with new db schema

Update 11/14 - reduced rdkit descriptor set (2022 instead of 2025)

In [1]:
import sys

import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm

notebookdir = Path.cwd().parent
sys.path.append(str(notebookdir))  # this way we can import src modules even in different subfolders

import sqlalchemy as sa
from sqlalchemy.orm import sessionmaker
from src.db_schema import *
from src.db_utils import get_basic_data, get_row_from_smiles, write_descriptors
from src.rdkit_tools import calculate_descriptors


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 250)

In [2]:
# set directories and filenames, load data, create db if not exists
working_dir = Path.cwd().parent
data_dir = working_dir / "processed_data"
database_file = data_dir / "hsbd_t_half_all.db"
engine = sa.create_engine(f"sqlite:///{database_file}")
Session = sessionmaker(bind=engine)

In [3]:
# get all basic data (reference, t_half, SMILES)
air_data = get_basic_data("air", Session)
soil_data = get_basic_data("soil", Session)
water_data = get_basic_data("water", Session)
sediment_data = get_basic_data("sediment", Session)

In [4]:
air_data = calculate_descriptors(air_data)
water_data = calculate_descriptors(water_data)
soil_data = calculate_descriptors(soil_data)
sediment_data = calculate_descriptors(sediment_data)

Calculating MACCS fingerprints: 100%|██████████| 347/347 [00:00<00:00, 3953.60it/s]


In [5]:
for index in tqdm(air_data.index, desc="Writing air descriptors"):
    row = air_data.loc[index]
    descriptor_dict = row.iloc[3:].to_dict()
    write_descriptors("air", row["reference"], descriptor_dict, Session)

for index in tqdm(water_data.index, desc="Writing water descriptors"):
    row = water_data.loc[index]
    descriptor_dict = row.iloc[3:].to_dict()
    write_descriptors("water", row["reference"], descriptor_dict, Session)

for index in tqdm(soil_data.index, desc="Writing soil descriptors"):
    row = soil_data.loc[index]
    descriptor_dict = row.iloc[3:].to_dict()
    write_descriptors("soil", row["reference"], descriptor_dict, Session)

for index in tqdm(sediment_data.index, desc="Writing sediment descriptors"):
    row = sediment_data.loc[index]
    descriptor_dict = row.iloc[3:].to_dict()
    write_descriptors("sediment", row["reference"], descriptor_dict, Session)

Writing air descriptors:   0%|          | 0/308 [00:00<?, ?it/s]

Writing water descriptors:   0%|          | 0/673 [00:00<?, ?it/s]

Writing soil descriptors:   0%|          | 0/672 [00:00<?, ?it/s]

Writing sediment descriptors:   0%|          | 0/347 [00:00<?, ?it/s]

In [6]:
# validation: read back some rows and print
row = get_row_from_smiles("soil", "CCO", Session)
print(row)

MACCS_052                 0.0
MACCS_053                 0.0
MACCS_054                 0.0
MACCS_055                 0.0
MACCS_056                 0.0
                       ...   
MaxAbsEStateIndex    7.569444
SlogP_VSA10               0.0
fr_C_O                    0.0
fr_pyridine               0.0
MACCS_051                 0.0
Length: 379, dtype: object
